# PathSnap: Exploratory Data Analysis & Spatial Indexing
## Phase 1 Review Deliverable - Member 1 (Data Engine, Spatial Indexing & EDA)

### Notebook Objectives:
1. **Parsers & Schema Validation**: Load `.track`, `.nodes`, `.arcs`, and `.route` files without assumption.
2. **Missing-Value & Quality Audit**: Verify data completeness, data types, and geographical coordinate plausibility.
3. **Temporal Dynamics**: Analyze the GPS sampling interval distribution, nominal 1 Hz rate, and temporal dropouts.
4. **Kinematics & Trajectory Extent**: Compute instantaneous velocities, acceleration, and total distance traveled.
5. **Road Network Graph Setup**: Build a directed road network (`networkx.DiGraph`) and analyze topology, connectivity, and arc lengths.
6. **Spatial Candidate Generation**: Implement and evaluate spatial candidate generation using `shapely.STRtree` and `scipy.spatial.KDTree`.
7. **Ground-Truth Labeling & Contract 1**: Chronologically align observations to the reference route and export `candidates_df` for Member 2 (Features & Models).
8. **Geographical Visualization**: Generate and inspect multi-layer interactive Folium maps.


In [ ]:
import os
import sys
from pathlib import Path

# Add project root to path
PROJECT_ROOT = Path("..").resolve() if Path("..").resolve().name == "ml-hackathon" else Path(".").resolve()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import networkx as nx
import folium

from src.data_loader import (
    load_trip,
    build_road_graph,
    haversine_distance,
    compute_bearing,
    wgs84_to_local_metric,
)
from src.candidate_gen import SpatialCandidateGenerator
from src.visualizer import create_folium_map, save_folium_map

# Styling
plt.style.use("seaborn-v0_8-whitegrid" if "seaborn-v0_8-whitegrid" in plt.style.available else "default")
plt.rcParams["figure.figsize"] = (10, 5)
plt.rcParams["font.size"] = 11

print("Environment and PathSnap modules initialized successfully!")


## 1. Raw Data Loading & File Inventory
We inspect trip `00000086` from the dataset. Each trip consists of:
- **`.track`**: GPS observations `[longitude, latitude, timestamp]`
- **`.nodes`**: Geographical nodes `[longitude, latitude]` with 0-based node index
- **`.arcs`**: Directed road segments `[from_node_id, to_node_id]` with 0-based arc index
- **`.route`**: Ground-truth sequence of traversed `arc_id`s


In [ ]:
trip_id = "00000086"
trip = load_trip(trip_id, base_dirs=["data", ".", "../data"])

track = trip["track"]
nodes = trip["nodes"]
arcs = trip["arcs"]
route = trip["route"]

print(f"=== Dataset Inventory for Trip {trip_id} ===")
print(f"GPS Track observations (.track) : {len(track):,} rows")
print(f"Road network nodes (.nodes)     : {len(nodes):,} nodes")
print(f"Road network arcs (.arcs)       : {len(arcs):,} directed arcs")
print(f"Reference route arcs (.route)   : {len(route):,} traversed arcs")


## 2. Missing-Value & Data Quality Audit
Before building any models, we perform a strict null-value and coordinate check.


In [ ]:
audit_data = [
    {"File": ".track", "Rows": len(track), "Nulls": track.isnull().sum().sum(), "Columns": list(track.columns)},
    {"File": ".nodes", "Rows": len(nodes), "Nulls": nodes.isnull().sum().sum(), "Columns": list(nodes.columns)},
    {"File": ".arcs", "Rows": len(arcs), "Nulls": arcs.isnull().sum().sum(), "Columns": list(arcs.columns)},
    {"File": ".route", "Rows": len(route), "Nulls": route.isnull().sum().sum(), "Columns": list(route.columns)},
]
audit_df = pd.DataFrame(audit_data)
display(audit_df)

print("Coordinate bounds check:")
print(f"Track Lon bounds: [{track['lon'].min():.6f}, {track['lon'].max():.6f}]")
print(f"Track Lat bounds: [{track['lat'].min():.6f}, {track['lat'].max():.6f}]")
print(f"Nodes Lon bounds: [{nodes['lon'].min():.6f}, {nodes['lon'].max():.6f}]")
print(f"Nodes Lat bounds: [{nodes['lat'].min():.6f}, {nodes['lat'].max():.6f}]")


## 3. GPS Sampling Interval & Dropout Analysis
The dataset documentation specifies that sampling is nominally 1 Hz. We calculate step intervals $\Delta t = t_i - t_{i-1}$ to identify irregular time gaps and signal dropouts.


In [ ]:
dt = track["dt"].dropna()
print(f"Total time intervals : {len(dt):,}")
print(f"Nominal 1.0s steps   : {(dt == 1.0).sum():,} ({(dt == 1.0).mean()*100:.2f}%)")
print(f"Minimum dt           : {dt.min():.2f} s")
print(f"Maximum dt (dropout) : {dt.max():.2f} s")
print(f"Mean dt              : {dt.mean():.4f} s")
print(f"Median dt            : {dt.median():.2f} s")

# Distribution plot
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 4.5))

top_dts = dt.value_counts().head(5)
labels = [f"{v:.1f}s" for v in top_dts.index]
percentages = (top_dts.values / len(dt)) * 100
bars = ax1.bar(labels, percentages, color="#2980b9", edgecolor="black", alpha=0.85)
ax1.set_title("Sampling Interval Distribution (Nominal 1 Hz)")
ax1.set_xlabel("Interval Delta (seconds)")
ax1.set_ylabel("Frequency (%)")
for bar in bars:
    yval = bar.get_height()
    ax1.text(bar.get_x() + bar.get_width()/2.0, yval + 1.0, f"{yval:.2f}%", ha="center", va="bottom")
ax1.set_ylim(0, 105)

ax2.plot(track["step_idx"], track["dt"], color="#e74c3c", lw=1.2, label="Observed dt")
ax2.axhline(1.0, color="#27ae60", ls="--", lw=1.5, label="Nominal 1.0s")
gaps = track[track["dt"] > 1.5]
if not gaps.empty:
    ax2.scatter(gaps["step_idx"], gaps["dt"], color="#c0392b", s=30, zorder=5, label=f"Gaps > 1.5s (n={len(gaps)})")
ax2.set_title("Temporal Continuity & GPS Signal Gaps")
ax2.set_xlabel("Sequential Step Index")
ax2.set_ylabel("Delta t (s)")
ax2.legend()

plt.tight_layout()
plt.show()


## 4. Trajectory Kinematics & Velocity Profile
We calculate step-wise Haversine displacement and derived vehicle speed (km/h) across the trajectory.


In [ ]:
lat1, lon1 = track["lat"].values[:-1], track["lon"].values[:-1]
lat2, lon2 = track["lat"].values[1:], track["lon"].values[1:]
step_dists = haversine_distance(lon1, lat1, lon2, lat2)
speeds_mps = step_dists / np.maximum(dt.values, 1e-3)
speeds_kmh = speeds_mps * 3.6
cum_dist_km = np.cumsum(np.insert(step_dists, 0, 0.0)) / 1000.0

total_duration_s = track["timestamp"].iloc[-1] - track["timestamp"].iloc[0]

print(f"Total Trajectory Length : {cum_dist_km[-1]:.2f} km")
print(f"Total Trajectory Time   : {total_duration_s:.1f} s ({total_duration_s / 60:.1f} min)")
print(f"Average Speed           : {speeds_kmh.mean():.2f} km/h")
print(f"Median Speed            : {np.median(speeds_kmh):.2f} km/h")
print(f"95th Percentile Speed   : {np.percentile(speeds_kmh, 95):.2f} km/h")
print(f"Max Speed               : {speeds_kmh.max():.2f} km/h")

fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(13, 6.5), sharex=True)
ax1.plot(track["step_idx"].values[1:], speeds_kmh, color="#34495e", lw=0.9, alpha=0.7, label="Instantaneous Speed (km/h)")
ma_speed = pd.Series(speeds_kmh).rolling(30, min_periods=1).mean()
ax1.plot(track["step_idx"].values[1:], ma_speed, color="#e67e22", lw=2.0, label="30-step Moving Average")
ax1.set_title(f"Vehicle Speed Profile (Trip {trip_id})")
ax1.set_ylabel("Speed (km/h)")
ax1.legend(loc="upper right")

ax2.plot(track["step_idx"], cum_dist_km, color="#16a085", lw=2.0, label="Cumulative Distance")
ax2.set_title("Trajectory Progress Over Time")
ax2.set_xlabel("Observation Step Index")
ax2.set_ylabel("Cumulative Distance (km)")
ax2.legend(loc="lower right")

plt.tight_layout()
plt.show()


## 5. Road Network Graph Construction & Topology Analysis
We build the directed road network using `networkx.DiGraph`.
- Nodes represent spatial intersections with angular (lon, lat) and metric (x, y) coordinates.
- Arcs represent one-way or two-way road segments with associated length (m) and bearing (degrees).


In [ ]:
G, aug_arcs = build_road_graph(nodes, arcs)

print(f"Graph Construction Complete:")
print(f"  Nodes : {G.number_of_nodes():,}")
print(f"  Edges : {G.number_of_edges():,}")

in_degrees = [d for n, d in G.in_degree()]
out_degrees = [d for n, d in G.out_degree()]
arc_lengths = aug_arcs["arc_length"].values

print(f"In-Degree   : Mean = {np.mean(in_degrees):.2f}, Max = {max(in_degrees)}")
print(f"Out-Degree  : Mean = {np.mean(out_degrees):.2f}, Max = {max(out_degrees)}")
print(f"Arc Lengths : Min = {arc_lengths.min():.1f}m, Median = {np.median(arc_lengths):.1f}m, Mean = {arc_lengths.mean():.1f}m, Max = {arc_lengths.max():.1f}m")

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 4.5))
bins = np.arange(0, 9) - 0.5
ax1.hist([in_degrees, out_degrees], bins=bins, label=["In-degree", "Out-degree"], color=["#3498db", "#9b59b6"], rwidth=0.8, density=True)
ax1.set_title(f"Node Degree Distribution (N = {G.number_of_nodes():,})")
ax1.set_xlabel("Degree")
ax1.set_ylabel("Relative Frequency")
ax1.set_xticks(range(0, 8))
ax1.legend()

ax2.hist(arc_lengths, bins=50, range=(0, 500), color="#2c3e50", edgecolor="white", density=True, alpha=0.85)
ax2.axvline(np.median(arc_lengths), color="#e74c3c", ls="--", lw=2, label=f"Median: {np.median(arc_lengths):.1f}m")
ax2.axvline(np.mean(arc_lengths), color="#f39c12", ls=":", lw=2, label=f"Mean: {np.mean(arc_lengths):.1f}m")
ax2.set_title(f"Arc Length Distribution (Total Arcs = {len(arc_lengths):,})")
ax2.set_xlabel("Arc Length (meters)")
ax2.set_ylabel("Density")
ax2.legend()

plt.tight_layout()
plt.show()


## 6. Spatial Candidate Generation & Recall Benchmarking
We utilize `SpatialCandidateGenerator` powered by `shapely.STRtree` (R-tree) and `scipy.spatial.KDTree` to retrieve the top $K$ nearest road arcs within a metric search radius $R = 100\text{ m}$.

We evaluate candidate recall:
$$\text{Recall@}K = \frac{\sum_{t=1}^N \mathbb{I}(\text{GroundTruthArc}_t \in \text{Candidates}_t)}{N}$$


In [ ]:
gen = SpatialCandidateGenerator(
    aug_arcs,
    origin_lon=float(nodes["lon"].mean()),
    origin_lat=float(nodes["lat"].mean()),
)

# Test candidate recall across different K values
cands_k10 = gen.generate_candidates(trip_id, track, route, k=10, max_distance=100.0)

for k_val in [3, 5, 8, 10]:
    sub_cands = cands_k10.groupby("step_idx").head(k_val)
    has_gt = sub_cands.groupby("step_idx")["label"].max()
    coverage = has_gt.mean() * 100.0
    miss_rate = 100.0 - coverage
    print(f"Top-{k_val:2d} Candidate Recall : {coverage:.2f}% | Candidate Miss Rate : {miss_rate:.2f}%")

print(f"\nTotal Candidate Pairs Generated (K=10): {len(cands_k10):,}")
print(f"Average Candidates per GPS Point: {len(cands_k10) / len(track):.2f}")
display(cands_k10.head(10))


## 7. Geometric Distance Separation
We inspect the perpendicular distance distribution of True Match candidate arcs vs Distractor candidate arcs.


In [ ]:
gt_cands = cands_k10[cands_k10["label"] == 1]
distractor_cands = cands_k10[cands_k10["label"] == 0]

print(f"Ground-Truth Arcs Distance: Mean = {gt_cands['dist_to_arc'].mean():.2f}m, Median = {gt_cands['dist_to_arc'].median():.2f}m, Max = {gt_cands['dist_to_arc'].max():.2f}m")
print(f"Distractor Arcs Distance  : Mean = {distractor_cands['dist_to_arc'].mean():.2f}m, Median = {distractor_cands['dist_to_arc'].median():.2f}m")

fig, ax = plt.subplots(figsize=(10, 4.5))
sns.kdeplot(gt_cands["dist_to_arc"], ax=ax, color="#27ae60", fill=True, label=f"Ground-Truth Arc (Mean: {gt_cands['dist_to_arc'].mean():.1f}m)")
sns.kdeplot(distractor_cands["dist_to_arc"], ax=ax, color="#e74c3c", fill=True, label=f"Distractor Arcs (Mean: {distractor_cands['dist_to_arc'].mean():.1f}m)")
ax.set_title("Perpendicular Distance Separation (True Arc vs Distractors)")
ax.set_xlabel("Perpendicular Distance to Arc (meters)")
ax.set_ylabel("Density")
ax.set_xlim(0, 100)
ax.legend()
plt.tight_layout()
plt.show()


## 8. Interactive Folium Geographical Map
We generate the multi-layer Folium map displaying:
- **Layer 1**: Raw GPS observations (blue circle markers & trajectory line)
- **Layer 2**: Relevant road network corridor (light gray lines)
- **Layer 3**: Ground-truth traversed route (green line)


In [ ]:
# Create Folium Map
folium_map = create_folium_map(
    track_df=track,
    nodes_df=nodes,
    arcs_df=arcs,
    route_df=route,
    candidate_arc_ids=cands_k10["arc_id"].unique(),
    title=f"PathSnap Trip {trip_id} Map Matching",
)

out_map_path = PROJECT_ROOT / "artifacts" / "maps" / f"trip_{trip_id}_map.html"
save_folium_map(folium_map, out_map_path)
print(f"Interactive Folium map saved to: {out_map_path}")

# Display map inline (or iframe)
folium_map


## 9. Contract 1 DataFrame Export (Handoff to Member 2)
We export the verified `candidates_df` following Contract 1 for immediate model training and feature extraction by Member 2.


In [ ]:
out_parquet = PROJECT_ROOT / "artifacts" / f"candidates_{trip_id}.parquet"
out_csv = PROJECT_ROOT / "artifacts" / f"candidates_{trip_id}.csv"

# Enforce Contract 1 schema
contract1_cols = [
    "trip_id",
    "step_idx",
    "timestamp",
    "gps_lon",
    "gps_lat",
    "arc_id",
    "from_node",
    "to_node",
    "arc_length",
    "label",
]

cands_k10[contract1_cols].to_csv(out_csv, index=False)
print(f"Contract 1 CSV exported: {out_csv} ({os.path.getsize(out_csv)/1e6:.2f} MB)")

# Verify schema and absence of nulls
assert list(cands_k10[contract1_cols].columns) == contract1_cols
assert cands_k10[contract1_cols].isnull().sum().sum() == 0
print("Contract 1 validation successful: Schema, column types, and integrity 100% verified!")
